# FIRMA DIGITAL EN PYTHON
## Objetivo: Comprender cómo se genera y verifica una firma digital

---

## PASO 1: Instalar cryptography

Primero, necesitas instalar la librería cryptography. En tu terminal, ejecuta:

```bash
$pip install cryptography

## PASO 2: Generar par de claves RSA
Vamos a generar un par de claves RSA (privada y pública):

In [6]:
from cryptography.hazmat.primitives.asymmetric import rsa

# Generar par de claves RSA
# - La clave privada sirve para firmar mensajes
# - La clave pública sirve para verificar la firma

private_key = rsa.generate_private_key(
    public_exponent=65537,  # Valor recomendado para RSA
    key_size=2048           # Tamaño de clave seguro actualmente
)

public_key = private_key.public_key()

print("✅ Par de claves RSA generado exitosamente")
print(f"Clave privada: {private_key}")
print(f"Clave pública: {public_key}")

✅ Par de claves RSA generado exitosamente
Clave privada: <cryptography.hazmat.bindings._rust.openssl.rsa.RSAPrivateKey object at 0x000001E07A36A210>
Clave pública: <cryptography.hazmat.bindings._rust.openssl.rsa.RSAPublicKey object at 0x000001E079088530>


## PASO 3: Firmar un mensaje
Ahora vamos a crear un mensaje y firmarlo con nuestra clave privada:

In [7]:
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import padding

# Mensaje original
mensaje = b"Este es un mensaje importante"

# Crear firma con la clave privada
# Se usa el algoritmo RSA-PSS con SHA-256
firma = private_key.sign(
    mensaje,
    padding.PSS(
        mgf=padding.MGF1(hashes.SHA256()),
        salt_length=padding.PSS.MAX_LENGTH
    ),
    hashes.SHA256()
)

print("📝 Mensaje original:", mensaje.decode())
print("🔐 Firma generada (hex):", firma.hex())
print(f"Tamaño de la firma: {len(firma)} bytes")

📝 Mensaje original: Este es un mensaje importante
🔐 Firma generada (hex): 436170c2c3144090c29271e0fbfa22d6ad62e4e3238931a7a0eeda801501193f6a0e245f9a5bdc308c7ae700dd6b519cac30504590495f0d2d2b40ad947f861f78bc077f9e20c11c8a9982b84d657a7fa2fd20aa45abe1cc6cb354ad4e311aad093db5a2beef8938e9a25f1a897e031a7c24fd39a114ce9627486448a464523fa8f85fd1e97e55c18653b5f1eb75a28496ddcb7836925bfd223fca3bca6286d734daef5088b4a254ed727bdda336387df8c24b9a76b48303670fdd4c1907e0b6e71500e1f35b55b80e581fcd55a0f9e67af692301fdedcb538a00136691d67d0b61d14ff24c8e76d0df4f636f7f6288cde11f0d01960d1134d75cd324a294aff
Tamaño de la firma: 256 bytes


## PASO 4: Verificar la firma
Verificamos que la firma es válida para el mensaje original:

In [8]:
print("\n--- VERIFICANDO FIRMA (mensaje original) ---")

try:
    public_key.verify(
        firma,
        mensaje,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )
    print("✅ La firma es válida (mensaje íntegro)")
except:
    print("❌ La firma no es válida")


--- VERIFICANDO FIRMA (mensaje original) ---
✅ La firma es válida (mensaje íntegro)


## PASO 5: Probar con mensaje modificado
Ahora modificamos el mensaje y vemos cómo falla la verificación:

In [10]:
# Mensaje modificado
mensaje_modificado = b"Este es un mensaje alterado"

print("\n--- VERIFICANDO FIRMA (mensaje MODIFICADO) ---")

try:
    public_key.verify(
        firma,
        mensaje_modificado,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )
    print("✅ La firma es válida (Esto NO debería pasar)")
except Exception as e:
    print("❌ La firma NO es válida (el mensaje fue modificado)")
    print("  ✓ Comportamiento esperado: la firma debe fallar")
    print(f"  📌 Detalle del error: {e}")


--- VERIFICANDO FIRMA (mensaje MODIFICADO) ---
❌ La firma NO es válida (el mensaje fue modificado)
  ✓ Comportamiento esperado: la firma debe fallar
  📌 Detalle del error: 
